# Termin 2 – Unsupervised Learning

## Einführung in PCA und KMeans am Iris-Datensatz

In diesem Notebook lernen wir zwei grundlegende Methoden des **unüberwachten Lernens** kennen:

1. **KMeans-Clustering** zur Gruppierung ähnlicher Datenpunkte  
2. **Principal Component Analysis (PCA)** zur Dimensionsreduktion und Visualisierung

Der Iris-Datensatz enthält Messwerte von Blüten. Die bekannten Arten werden in diesem Notebook **nicht zum Trainieren** verwendet.  
Sie dienen nur am Ende zur Visualisierung und zur didaktischen Einordnung.


## Lernziele

Nach dieser Übung solltet ihr:

- erklären können, was unüberwachtes Lernen von überwachtem Lernen unterscheidet,
- den Unterschied zwischen **Merkmalen** und **Labels** beschreiben können,
- KMeans-Clustering anwenden und die Rolle der Clusteranzahl `k` verstehen,
- PCA zur Reduktion von vier Merkmalen auf zwei oder drei Hauptkomponenten nutzen,
- interne Clustering-Metriken interpretieren,
- verstehen, warum verschiedene Metriken unterschiedliche optimale Clusteranzahlen vorschlagen können.


## 1. Vorbereitung


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
)

RANDOM_STATE = 42


## 2. Datensatz laden

Wir verwenden den Iris-Datensatz aus `scikit-learn`.

Wichtig für unüberwachtes Lernen:

- `X` enthält die **Merkmale** und wird für PCA und KMeans verwendet.
- `target` enthält die bekannten Iris-Arten. Diese Information wird **nicht zum Trainieren** genutzt.
- `target` ist hier nur hilfreich, um die Ergebnisse später mit der bekannten Struktur zu vergleichen.


In [ ]:
iris = load_iris()

X = iris.data
target = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df_iris = pd.DataFrame(X, columns=feature_names)
df_iris["target"] = target
df_iris["species"] = [target_names[i] for i in target]

print("Form der Merkmalsmatrix X:", X.shape)
print("Anzahl der Klassen:", len(target_names))

df_iris.head()


## 3. Erste visuelle Exploration

Da der Datensatz vier numerische Merkmale enthält, können wir alle paarweisen 2D-Kombinationen darstellen.

Dabei verwenden wir zunächst **keine Farbcodierung nach Arten**.  
Das entspricht eher einer echten unüberwachten Situation, in der die Labels nicht bekannt sind.


In [ ]:
def plot_feature_pairs(X, feature_names):
    pairs = list(combinations(range(X.shape[1]), 2))

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()

    for ax, (i, j) in zip(axes, pairs):
        ax.scatter(X[:, i], X[:, j], s=45, alpha=0.75)
        ax.set_xlabel(feature_names[i])
        ax.set_ylabel(feature_names[j])
        ax.set_title(f"{feature_names[i]} vs. {feature_names[j]}")
        ax.grid(True, alpha=0.3)

    fig.suptitle("Iris-Datensatz: paarweise Merkmalskombinationen ohne Labels", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_feature_pairs(X, feature_names)


<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Kurze Zwischenfrage

Welche Merkmalskombinationen zeigen bereits visuell mögliche Gruppen?

Sind alle Gruppen gleich gut getrennt?

</div>

## 4. Warum Skalierung wichtig ist

PCA und KMeans arbeiten mit Abständen beziehungsweise Varianz.  
Daher können Merkmale mit größeren Zahlenbereichen das Ergebnis stärker beeinflussen.

Beim Iris-Datensatz sind alle Merkmale zwar in Zentimetern angegeben, aber eine Skalierung ist trotzdem eine gute Standardpraxis.  
Wir verwenden hier den `StandardScaler`, sodass jedes Merkmal ungefähr Mittelwert 0 und Standardabweichung 1 hat.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=feature_names)
df_scaled.head()


## 5. KMeans-Clustering

KMeans teilt Datenpunkte in `k` Cluster ein.  
Jedes Cluster wird durch ein Zentrum, den sogenannten **Zentroiden**, beschrieben.

Ablauf vereinfacht:

1. Wähle `k` Start-Zentren.
2. Weise jeden Datenpunkt dem nächsten Zentrum zu.
3. Berechne neue Zentren.
4. Wiederhole die Schritte, bis sich kaum noch etwas ändert.

Hier setzen wir zunächst `k = 3`, weil wir beim Iris-Datensatz wissen, dass es drei Arten gibt.  
In echten Anwendungen ist `k` oft unbekannt und muss abgeschätzt werden.


In [ ]:
# KMeans auf skalierten Daten
kmeans_scaled = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
cluster_scaled = kmeans_scaled.fit_predict(X_scaled)

df_iris["cluster_k3_scaled"] = cluster_scaled


# KMeans auf unskalierten Originaldaten
kmeans_raw = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
cluster_raw = kmeans_raw.fit_predict(X)

df_iris["cluster_k3_raw"] = cluster_raw


df_iris.head()


<div style="background-color:#fff3e6; border-left:5px solid #f4a261; padding:14px 18px; border-radius:6px;">

### Optionaler Vergleich der Clusterings

Der **Adjusted Rand Index** vergleicht zwei Cluster-Einteilungen.

Wir nutzen ihn, um zu prüfen, wie ähnlich das KMeans-Ergebnis auf skalierten und unskalierten Daten ist.

- Wert nahe **1**: sehr ähnlich
- Wert nahe **0**: kaum besser als zufällig

</div>

In [ ]:
from sklearn.metrics import adjusted_rand_score

adjusted_rand_score(
    df_iris["cluster_k3_scaled"],
    df_iris["cluster_k3_raw"]
)

### Visualisierung des KMeans-Ergebnisses in den Originalmerkmalen

Die Cluster wurden auf den skalierten Daten berechnet.  
Zur Darstellung verwenden wir wieder die originalen Merkmalsachsen, weil sie leichter interpretierbar sind.


In [ ]:
def plot_kmeans_feature_pairs(X_original, labels, feature_names):
    pairs = list(combinations(range(X_original.shape[1]), 2))

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()

    for ax, (i, j) in zip(axes, pairs):
        scatter = ax.scatter(
            X_original[:, i],
            X_original[:, j],
            c=labels,
            cmap="viridis",
            s=45,
            alpha=0.80,
        )
        ax.set_xlabel(feature_names[i])
        ax.set_ylabel(feature_names[j])
        ax.set_title(f"{feature_names[i]} vs. {feature_names[j]}")
        ax.grid(True, alpha=0.3)

    fig.suptitle("KMeans-Clustering mit k = 3", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_kmeans_feature_pairs(X, cluster_scaled, feature_names)
#plot_kmeans_feature_pairs(X, cluster_raw, feature_names)

## 6. Clusteranzahl bestimmen

Die Clusteranzahl `k` ist ein zentraler Parameter von KMeans.  
Wir testen deshalb mehrere Werte für `k` und berechnen verschiedene interne Bewertungsmetriken.

### Metriken


<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Aufgabe

Informiere dich in der **scikit-learn-Dokumentation** über die folgenden Clustering-Metriken:

- Inertia / Elbow-Methode
- Silhouette Score
- Calinski-Harabasz Index
- Davies-Bouldin Index

Trage anschließend für jede Metrik ein:

| Metrik | Ist ein hoher oder niedriger Wert besser? | Bestes `k` in unserem Beispiel | Kurze Begründung |
|---|---|---|---|
| Inertia / Elbow |  |  |  |
| Silhouette Score |  |  |  |
| Calinski-Harabasz Index |  |  |  |
| Davies-Bouldin Index |  |  |  |

Welche Clusteranzahl würdest du insgesamt wählen?

</div>
</div>

In [ ]:
def evaluate_kmeans(X_input, k_range=range(2, 10)):
    rows = []

    for k in k_range:
        model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
        labels = model.fit_predict(X_input)

        rows.append({
            "k": k,
            "Inertia": model.inertia_,
            "Silhouette Score": silhouette_score(X_input, labels),
            "Calinski-Harabasz Index": calinski_harabasz_score(X_input, labels),
            "Davies-Bouldin Index": davies_bouldin_score(X_input, labels),
        })

    return pd.DataFrame(rows)


df_scores_scaled = evaluate_kmeans(X_scaled)
df_scores_scaled

def plot_metric(df_scores, y_col, title, ylabel):
    plt.figure(figsize=(7, 4))
    plt.plot(df_scores["k"], df_scores[y_col], marker="o")
    plt.xlabel("Anzahl der Cluster (k)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(df_scores["k"])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_metric(df_scores_scaled, "Inertia", "Elbow-Methode: Inertia", "Inertia")
plot_metric(df_scores_scaled, "Silhouette Score", "Silhouette Score", "Silhouette Score")
plot_metric(df_scores_scaled, "Calinski-Harabasz Index", "Calinski-Harabasz Index", "Calinski-Harabasz Index")
plot_metric(df_scores_scaled, "Davies-Bouldin Index", "Davies-Bouldin Index", "Davies-Bouldin Index")


### Interpretation der Metriken

Die Metriken müssen nicht alle denselben Wert für `k` bevorzugen.

Beim Iris-Datensatz ist häufig zu sehen:

- `k = 2` trennt vor allem **Setosa** von den beiden stärker überlappenden Arten.
- `k = 3` passt besser zur bekannten biologischen Einteilung in drei Arten.
- Interne Metriken kennen die echten Arten nicht und bewerten nur die geometrische Struktur der Daten.

Das ist ein wichtiger Punkt:  
**Unsupervised Learning findet Strukturen in den Daten, aber diese Strukturen müssen fachlich interpretiert werden.**


## 7. PCA: Dimensionsreduktion

PCA transformiert korrelierte ursprüngliche Merkmale in neue Achsen, die sogenannten **Hauptkomponenten**.

Eigenschaften:

- `PC1` erklärt möglichst viel Varianz.
- `PC2` erklärt möglichst viel der verbleibenden Varianz und steht senkrecht zu `PC1`.
- Weitere PCs erklären jeweils weitere Varianzanteile.
- Die Zielvariable wird für die PCA **nicht** verwendet.

Wir wenden PCA auf die skalierten Merkmale an.


In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

df_explained = pd.DataFrame({
    "Komponente": [f"PC{i}" for i in range(1, len(explained) + 1)],
    "Erklärte Varianz": explained,
    "Kumulierte erklärte Varianz": cumulative,
})

df_explained


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.xlabel("Anzahl der Hauptkomponenten")
plt.ylabel("Kumulierte erklärte Varianz")
plt.title("PCA: kumulierte erklärte Varianz")
plt.xticks(range(1, len(cumulative) + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. PCA auf zwei und drei Komponenten

Für eine 2D-Visualisierung sind zwei Hauptkomponenten besonders praktisch.  
Drei Hauptkomponenten können etwas mehr Information erhalten, sind aber schwerer darzustellen.


In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3d = pca_3d.fit_transform(X_scaled)

df_pca_2d = pd.DataFrame(X_pca_2d, columns=["PC1", "PC2"])
df_pca_2d["species"] = [target_names[i] for i in target]

df_pca_3d = pd.DataFrame(X_pca_3d, columns=["PC1", "PC2", "PC3"])
df_pca_3d["species"] = [target_names[i] for i in target]

print("Originale Daten:", X.shape)
print("PCA 2D:", X_pca_2d.shape)
print("PCA 3D:", X_pca_3d.shape)

df_pca_2d.head()


### PCA-Visualisierung mit bekannten Arten

Die Farben zeigen hier die bekannten Iris-Arten.  
Das ist **keine Trainingsinformation**, sondern nur eine nachträgliche Visualisierung.


In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_pca_2d["PC1"],
    df_pca_2d["PC2"],
    c=target,
    cmap="viridis",
    s=50,
    alpha=0.85,
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA-Projektion des Iris-Datensatzes")
plt.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(target_names)
cbar.set_label("Iris-Art")

plt.tight_layout()
plt.show()


<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Aufgabe: Kombination von PCA und KMeans

Bisher haben wir zwei Methoden getrennt betrachtet:

- **PCA** zur Reduktion der Merkmalsdimension
- **KMeans** zur Gruppierung ähnlicher Datenpunkte

In dieser Aufgabe kombinierst du beide Ansätze.

Die Idee ist:

1. Die ursprünglichen vier Iris-Merkmale werden mit PCA auf zwei Hauptkomponenten reduziert.
2. Anschließend wird KMeans auf diesen zwei PCA-Komponenten angewendet.
3. Das Ergebnis wird im PCA-Raum visualisiert.

Bearbeite die folgenden Schritte im Code:

1. Wende KMeans auf die PCA-transformierten Daten an.
2. Verwende zunächst `k = 3`.
3. Speichere die vorhergesagten Clusterlabels.
4. Erstelle einen Scatterplot der beiden PCA-Komponenten und färbe die Punkte nach den KMeans-Clustern ein.
5. Vergleiche das Ergebnis mit der bekannten Iris-Art.

</div>

In [ ]:
# PCA und KMeans kombinieren

# 1. PCA mit zwei Hauptkomponenten definieren


# 2. PCA auf die skalierten Daten anwenden


# 3. PCA-Ergebnis als DataFrame speichern


# Optional: echte Iris-Arten ergänzen, falls später verglichen werden soll


# 4. KMeans-Modell mit k = 3 definieren

# 5. KMeans auf die PCA-Daten anwenden


# 6. Ergebnis anzeigen


<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Aufgabe: KMeans-Cluster im PCA-Raum visualisieren

Visualisiere nun das Ergebnis des KMeans-Clusterings im PCA-Raum.

Dabei gilt:

- x-Achse: `PC1`
- y-Achse: `PC2`
- Farbe: von KMeans gefundene Cluster
- Titel: `"KMeans-Clustering im PCA-Raum"`

Überlege beim Betrachten des Plots:

- Sind die Cluster klar getrennt?
- Gibt es Bereiche, in denen Cluster überlappen?
- Sieht das Ergebnis plausibel aus?

</div>

In [ ]:
# KMeans-Cluster im PCA-Raum visualisieren



<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Aufgabe: Vergleich mit den bekannten Iris-Arten

KMeans hat die echten Iris-Arten nicht verwendet.  
Für die Interpretation können wir sie aber nachträglich mit den gefundenen Clustern vergleichen.

Erstelle dazu einen zweiten Plot im PCA-Raum.

Dabei gilt:

- x-Achse: `PC1`
- y-Achse: `PC2`
- Farbe: bekannte Iris-Art
- Titel: `"Bekannte Iris-Arten im PCA-Raum"`

Vergleiche anschließend diesen Plot mit dem vorherigen KMeans-Plot.

</div>

In [ ]:
# Bekannte Iris-Arten im PCA-Raum visualisieren



<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Aufgabe: Cluster und Arten mit einer Kreuztabelle vergleichen

Die Plots geben einen visuellen Eindruck davon, ob die gefundenen KMeans-Cluster zu den bekannten Iris-Arten passen.

Zusätzlich können wir den Vergleich tabellarisch darstellen. Dafür verwenden wir eine sogenannte **Kreuztabelle**.

Eine Kreuztabelle zählt, wie häufig Kombinationen aus zwei Kategorien vorkommen.  
In unserem Fall vergleichen wir:

- die **bekannten Iris-Arten**
- die von **KMeans gefundenen Cluster**

Die Kreuztabelle zeigt also zum Beispiel, wie viele Datenpunkte der Art `setosa` in Cluster `0`, Cluster `1` oder Cluster `2` gelandet sind.

In `pandas` kannst du eine Kreuztabelle mit `pd.crosstab()` erstellen:

```python
pd.crosstab(zeilen_variable, spalten_variable)

In [ ]:
# Kreuztabelle: bekannte Arten vs. KMeans-Cluster


<div style="background-color:#eef6ff; border-left:5px solid #2f80ed; padding:14px 18px; border-radius:6px;">

### Interpretation: Was zeigt die Kreuztabelle?

Die Kreuztabelle zeigt, wie gut die von KMeans gefundenen Cluster mit den bekannten Iris-Arten übereinstimmen.

Lies die Tabelle zeilenweise:

- Jede Zeile steht für eine bekannte Iris-Art.
- Jede Spalte steht für einen von KMeans gefundenen Cluster.
- Die Zahlen zeigen, wie viele Datenpunkte einer Art in einem bestimmten Cluster liegen.

Ein gutes Clustering wäre daran erkennbar, dass jede Iris-Art möglichst stark in nur einem Cluster vertreten ist.

Beantworte nun:

1. Welche Iris-Art wird besonders eindeutig einem Cluster zugeordnet?
2. Welche Arten werden weniger klar getrennt?
3. Entspricht jedes KMeans-Cluster genau einer Iris-Art?
4. Was sagt das über die Grenzen von KMeans aus?

</div>